# Workshop 6 — Transfer Learning with Pre-trained CNNs
**Saharsh Pathak | 2417371 | Herald College Kathmandu**

Using VGG16 and MobileNetV2 as feature extractors for image classification.
Transfer learning allows us to leverage models trained on ImageNet (1.2M images, 1000 classes).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Transfer Learning Concept

**Why Transfer Learning?**
- Training deep CNNs from scratch requires millions of images and days of compute
- Pre-trained models already learned low-level features (edges, textures, shapes)
- We only need to train the final classification head
- Achieves high accuracy with small datasets

## 2. Feature Extraction with VGG16

In [ ]:
IMG_SIZE = 224
NUM_CLASSES = 10  # Adjust for your dataset

def build_vgg16_transfer(num_classes=10, fine_tune_layers=0):
    """
    VGG16 as feature extractor.
    fine_tune_layers: number of top VGG16 layers to unfreeze for fine-tuning.
    """
    # Load VGG16 without top classification layers
    base = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

    # Freeze all base layers
    base.trainable = False

    # Optionally unfreeze top layers for fine-tuning
    if fine_tune_layers > 0:
        for layer in base.layers[-fine_tune_layers:]:
            layer.trainable = True

    # Build model
    model = keras.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='VGG16_Transfer')

    return model

vgg_model = build_vgg16_transfer(NUM_CLASSES)
vgg_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
vgg_model.summary()

trainable = sum(1 for l in vgg_model.layers if l.trainable)
total = len(vgg_model.layers)
print(f'Trainable layers: {trainable}/{total}')

## 3. MobileNetV2 — Lightweight Alternative

In [ ]:
def build_mobilenet_transfer(num_classes=10):
    """
    MobileNetV2 — 3.4M params vs VGG16's 138M.
    Ideal for mobile/edge deployment.
    """
    base = MobileNetV2(weights='imagenet', include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False

    model = keras.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='MobileNetV2_Transfer')

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

mobile_model = build_mobilenet_transfer(NUM_CLASSES)
print(f'MobileNetV2 params: {mobile_model.count_params():,}')
print(f'VGG16 params:       {vgg_model.count_params():,}')

## 4. Data Augmentation Pipeline

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

# Uncomment when dataset is available:
# train_gen = train_datagen.flow_from_directory('data/train', target_size=(224,224), batch_size=32, class_mode='categorical', subset='training')
# val_gen = train_datagen.flow_from_directory('data/train', target_size=(224,224), batch_size=32, class_mode='categorical', subset='validation')
print('Data pipeline configured')

## 5. Training Strategy: Feature Extraction → Fine-Tuning

In [ ]:
# Phase 1: Feature extraction (frozen base)
# Phase 2: Fine-tuning (unfreeze top layers)

def fine_tune_model(model, base_model_name, unfreeze_from_layer):
    """Unfreeze top layers of base model for fine-tuning."""
    for layer in model.layers:
        if hasattr(layer, 'layers'):  # it's the base model
            for i, l in enumerate(layer.layers):
                l.trainable = i >= unfreeze_from_layer

    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),  # lower LR for fine-tuning
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print('Fine-tuning strategy:')
print('  Phase 1: Train only classification head (10-20 epochs)')
print('  Phase 2: Unfreeze top 4 conv blocks, train with LR=1e-5 (10 epochs)')

## 6. Simulated Results Comparison

In [ ]:
np.random.seed(42)
epochs = 20

scratch = np.clip(np.linspace(0.15, 0.72, epochs) + np.random.normal(0, 0.03, epochs), 0, 1)
vgg_feat = np.clip(np.linspace(0.55, 0.89, epochs) + np.random.normal(0, 0.02, epochs), 0, 1)
vgg_fine = np.clip(np.linspace(0.70, 0.95, epochs) + np.random.normal(0, 0.015, epochs), 0, 1)
mobile = np.clip(np.linspace(0.60, 0.92, epochs) + np.random.normal(0, 0.02, epochs), 0, 1)

plt.figure(figsize=(10, 5))
plt.plot(scratch, label='From Scratch', color='#6B7280', linewidth=2, linestyle=':')
plt.plot(vgg_feat, label='VGG16 Feature Extraction', color='#0EA5E9', linewidth=2)
plt.plot(vgg_fine, label='VGG16 Fine-Tuned', color='#6366F1', linewidth=2)
plt.plot(mobile, label='MobileNetV2', color='#10B981', linewidth=2)
plt.title('Transfer Learning: Validation Accuracy Comparison')
plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print('Summary:')
print(f'  From Scratch:          ~{scratch[-1]:.2f}')
print(f'  VGG16 Feature Extract: ~{vgg_feat[-1]:.2f}')
print(f'  VGG16 Fine-Tuned:      ~{vgg_fine[-1]:.2f}')
print(f'  MobileNetV2:           ~{mobile[-1]:.2f}')